# Example Gamma Family Surface

Canonical gamma-family notebook covering incomplete gamma and related routed API metadata.

## Scope

This notebook is the canonical example surface for `example_gamma_family_surface`. It runs against the repo source tree through `/src`, shows direct public API usage, summarizes validation and benchmark status, and includes visual summaries.

In [ ]:
import io
import json
import os
import re
import subprocess
import sys
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'src' / 'arbplusjax').exists():
            return p
    raise RuntimeError(f'Could not locate repo root from: {start}')

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
os.chdir(REPO_ROOT)

PYTHON = os.getenv('ARBPLUSJAX_PYTHON', sys.executable)
JAX_MODE = os.getenv('JAX_MODE', 'cpu').strip().lower()
JAX_DTYPE = os.getenv('JAX_DTYPE', 'float64').strip().lower()
RUN_ENV = os.environ.copy()
RUN_ENV['PYTHONPATH'] = str(REPO_ROOT / 'src') + os.pathsep + RUN_ENV.get('PYTHONPATH', '')
if JAX_MODE == 'cpu':
    RUN_ENV['JAX_PLATFORMS'] = 'cpu'
elif JAX_MODE == 'gpu':
    RUN_ENV['JAX_PLATFORMS'] = 'cuda'
RUN_ENV['JAX_ENABLE_X64'] = '1' if JAX_DTYPE == 'float64' else '0'
EXAMPLE_INPUT_ROOT = REPO_ROOT / 'examples' / 'inputs' / 'example_gamma_family_surface'
EXAMPLE_OUTPUT_ROOT = REPO_ROOT / 'examples' / 'outputs' / 'example_gamma_family_surface'
EXAMPLE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def run(cmd: list[str], *, capture: bool = False):
    print('[cmd]', ' '.join(cmd))
    return subprocess.run(cmd, cwd=REPO_ROOT, env=RUN_ENV, text=True, capture_output=capture, check=True)


## Environment

The notebook reports interpreter, selected JAX mode, and the active backend/device view. Canonical retained execution in this repo state is CPU-oriented, but the notebook calling pattern remains CPU/GPU portable and explicitly parameterized for `float32` and `float64`.

In [ ]:
SUPPORTED_JAX_MODES = ('cpu', 'gpu')
SUPPORTED_JAX_DTYPES = ('float32', 'float64')
if JAX_MODE not in SUPPORTED_JAX_MODES:
    raise ValueError(f'Unsupported JAX_MODE: {JAX_MODE}')
if JAX_DTYPE not in SUPPORTED_JAX_DTYPES:
    raise ValueError(f'Unsupported JAX_DTYPE: {JAX_DTYPE}')
print('python:', PYTHON)
print('jax_mode:', JAX_MODE)
print('jax_dtype:', JAX_DTYPE)
print('supported_jax_modes:', SUPPORTED_JAX_MODES)
print('supported_jax_dtypes:', SUPPORTED_JAX_DTYPES)
print('validation_slice:', 'cpu_current__gpu_portable_contract')
runtime = run([PYTHON, 'tools/check_jax_runtime.py'], capture=True)
print(runtime.stdout)
runtime_payload = json.loads(runtime.stdout)
(EXAMPLE_OUTPUT_ROOT / f'runtime_{JAX_MODE}.json').write_text(json.dumps(runtime_payload, indent=2) + '\n', encoding='utf-8')

## Direct Usage

Exercise lower/upper incomplete gamma surfaces, complement identities, and diagnostics.

In [ ]:
import jax.numpy as jnp
from arbplusjax import api

s = jnp.asarray([1.5, 2.5, 3.5], dtype=jnp.float64)
z = jnp.asarray([0.75, 1.25, 1.75], dtype=jnp.float64)
gamma_results = {
    'upper_point': api.eval_point_batch('incomplete_gamma_upper', s, z, method='quadrature', regularized=True),
    'lower_point': api.eval_point_batch('incomplete_gamma_lower', s, z, method='quadrature', regularized=True),
    'upper_basic': api.eval_interval_batch('incomplete_gamma_upper', s, z, mode='basic', method='quadrature', regularized=True, prec_bits=53),
    'metadata': api.get_public_function_metadata('incomplete_gamma_upper'),
}
value, diagnostics = api.incomplete_gamma_upper(jnp.float64(0.75), jnp.float64(0.05), mode='point', method='auto', return_diagnostics=True)
display(gamma_results)
display({'diagnostic_method': diagnostics.method, 'fallback_used': diagnostics.fallback_used, 'value': value})

## Production Pattern

Gamma-family service calls should bind the method and precision policy up front and then reuse the bound callable over stable batch shapes. Diagnostics are optional but should be sampled in staging so fallback behavior is visible before deployment.

In [ ]:
gamma_bound = api.bind_point_batch('incomplete_gamma_upper', dtype='float64', pad_to=8, method='quadrature', regularized=True)
gamma_interval_bound = api.bind_interval_batch('incomplete_gamma_upper', mode='basic', dtype='float64', pad_to=8, prec_bits=53, method='quadrature', regularized=True)
gamma_service = {
    'point_bound': gamma_bound(s, z),
    'interval_bound': gamma_interval_bound(s, z),
}
display(gamma_service)

## Extending Benchmarks

To benchmark more gamma-family functions, extend the representative operation list in `benchmark_special_function_service_api.py` or add a dedicated comparison/benchmark entrypoint if the function has special reference needs.

## Fast JAX Point Pattern

Special-function fast JAX at the API level should use the compiled point-batch binder on a safe box with fixed method policy.

In [ ]:
import jax
gamma_fast = api.bind_point_batch_jit('incomplete_gamma_upper', dtype='float64', pad_to=8, method='quadrature', regularized=True)
gamma_fast_out = gamma_fast(s, z)
gamma_vmap = jax.vmap(lambda s_i, z_i: api.eval_point('incomplete_gamma_upper', s_i, z_i, dtype='float64', method='quadrature', regularized=True))(s, z)
display({'jit_shape': gamma_fast_out.shape, 'jit_matches_vmap': bool(jnp.allclose(gamma_fast_out, gamma_vmap, rtol=1e-6, atol=1e-6))})

## AD Product Pattern

Special-function AD should be shown on the production-facing differentiated surface with explicit method policy. This section validates incomplete-gamma derivatives over a sweep and plots primal versus gradient.

In [ ]:
import jax
s_fixed = jnp.asarray(2.5, dtype=jnp.float64)
def gamma_loss(zv):
    return api.incomplete_gamma_upper(s_fixed, zv, mode='point', method='quadrature')
z_sweep = jnp.linspace(0.25, 2.0, 32, dtype=jnp.float64)
primal_vals = jax.vmap(gamma_loss)(z_sweep)
grad_vals = jax.vmap(jax.grad(gamma_loss))(z_sweep)
ad_df = pd.DataFrame({'z': np.asarray(z_sweep), 'primal': np.asarray(primal_vals), 'grad': np.asarray(grad_vals)})
display(ad_df.head())
ax = ad_df.plot(x='z', y=['primal', 'grad'], figsize=(8, 4), title='Gamma AD Validation')
plt.tight_layout()
plt.savefig(EXAMPLE_OUTPUT_ROOT / f'ad_validation_{JAX_MODE}.png', dpi=160, bbox_inches='tight')
plt.show()

## Validation Summary

Run the incomplete-gamma and hardening tests.

In [ ]:
tests = run([
    PYTHON, '-m', 'pytest', '-q',
    'tests/test_incomplete_gamma.py',
    'tests/test_gamma_hardening.py',
    'tests/test_api_metadata.py',
], capture=True)
print(tests.stdout)
if tests.stderr:
    print(tests.stderr)
(EXAMPLE_OUTPUT_ROOT / f'pytest_{JAX_MODE}.txt').write_text(tests.stdout + ('\n' + tests.stderr if tests.stderr else ''), encoding='utf-8')

## Benchmark / Comparison Summary

Use the existing gamma comparison benchmarks when local reference libraries are available.

In [ ]:
compare_cmds = [
    [PYTHON, 'benchmarks/benchmark_gamma_compare.py', '--arb-repo', str(REPO_ROOT), '--iters', '128'],
    [PYTHON, 'benchmarks/benchmark_loggamma_compare.py', '--arb-repo', str(REPO_ROOT), '--iters', '128'],
]
rows = []
for cmd in compare_cmds:
    try:
        completed = run(cmd, capture=True)
        print(completed.stdout)
        rows.append({'script': Path(cmd[1]).name, 'status': 'ok', 'tail': completed.stdout[-2000:]})
    except subprocess.CalledProcessError as exc:
        print(exc.stdout)
        print(exc.stderr)
        rows.append({'script': Path(cmd[1]).name, 'status': 'failed_or_unavailable', 'tail': ((exc.stdout or '') + '\n' + (exc.stderr or ''))[-2000:]})
bench_df = pd.DataFrame(rows)
bench_df.to_csv(EXAMPLE_OUTPUT_ROOT / f'gamma_compare_status_{JAX_MODE}.csv', index=False)
display(bench_df)

## Optional Diagnostics

This tranche relies on the function-returned diagnostics objects rather than a separate compile-diagnostics benchmark.

In [ ]:
summary_lines = [
    f'# Example Gamma Family Surface Summary ({JAX_MODE})',
    '',
    f'- backend: `{runtime_payload["platform"]}`',
    f'- comparison_rows: `{len(bench_df)}`',
    f'- auto_method: `{diagnostics.method}`',
    '',
    '## Comparison Scripts',
    '',
]
for row in bench_df.to_dict(orient='records'):
    summary_lines.append(f"- `{row['script']}`: `{row['status']}`")
(EXAMPLE_OUTPUT_ROOT / f'summary_{JAX_MODE}.md').write_text('\n'.join(summary_lines) + '\n', encoding='utf-8')
display('\n'.join(summary_lines[:12]))